<a href="https://colab.research.google.com/github/SEAFARI/pytorch-deeplearning/blob/main/07_Pytorch_Experiment_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 07 Pytorch Experiment Tracking

ML is very experimental.

In helps to figure out what dosent work and thus it enables us to figure out what works.


In [1]:
import torch
import torchvision

print(f"Torch version: {torch.__version__} \nTorchvision version: {torchvision.__version__}")

Torch version: 2.9.0+cu126 
Torchvision version: 0.24.0+cu126


In [2]:
## Regular imports
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms

try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't Find torchinfo.summary, downloading....")
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from going_modular.going_modular import data_setup, engine
except:
  # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

[INFO] Couldn't Find torchinfo.summary, downloading....
[INFO] Couldn't find going_modular scripts... downloading them from GitHub.
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 4393, done.
remote: Total 4393 (delta 0), reused 0 (delta 0), pack-reused 4393 (from 1)
Receiving objects: 100% (4393/4393), 764.14 MiB | 16.52 MiB/s, done.
Resolving deltas: 100% (2657/2657), done.
Updating files: 100% (248/248), done.


In [3]:
## device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
def set_seeds(seed: int=42):
  """Sets random seed for torch operations.

  Args:
      seed(int, optional) : Random Seed to set. Defaults to 42.
  """
  ## set seed for general torch operations
  torch.manual_seed(seed)

  ## set seed for cuda operations
  torch.cuda.manual_seed(seed)

In [5]:
set_seeds()

## 1. Get data

Getting the pizza, steak, sushi data

so we can run model on FoodVisionMini and see which model performs the best.

In [6]:
from torch.utils import data
import os
import zipfile

from pathlib import Path
import requests

def download_data(source: str,
                  destination: str,
                  remove_source: bool= True) -> Path:
    ## setup data path
    data_path = Path("data/")
    image_path = data_path / destination

    ## if image folder dosent exist then downlaod from github
    if image_path.is_dir():
      print(f"{image_path} directory exists. Skipping re-download....")
    else:
      print(f"Couldn't find {image_path}, Downloading it....")
      image_path.mkdir(parents=True, exist_ok=True)

      ## Download from github
      target_file = Path(source).name
      with open(data_path/target_file , "wb") as f:
        request = requests.get(source)
        print("Downloading pizza_steak_sushi data.... ")
        f.write(request.content)

      ## unzip pizza_steak_sushi data
      with zipfile.ZipFile(data_path / target_file , "r") as zip_ref:
        print("Unzipping pizza steak sushi data")
        zip_ref.extractall(image_path)


      ## Remove the zip file
      if remove_source:
        os.remove(data_path/target_file )

    return image_path

In [7]:
download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
              destination="pizza_steak_sushi")

Couldn't find data/pizza_steak_sushi, Downloading it....
Unzipping pizza steak sushi data


PosixPath('data/pizza_steak_sushi')

## 2. Creating the dataloaders

In [13]:
image_path = Path("data/pizza_steak_sushi")
train_dir = image_path / "train"
test_dir = image_path/"test"

In [16]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

### 2.1. Manual Dataloader

In [25]:
from torchvision import transforms

manual_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [26]:
from going_modular.going_modular import data_setup

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transform,
    batch_size=32,
)


In [27]:
train_dataloader, test_dataloader, class_names


(<torch.utils.data.dataloader.DataLoader at 0x7ae22fa0f440>,
 ['pizza', 'steak', 'sushi'])

### 2.1. Automatic Dataloader

In [28]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
auto_transforms = weights.transforms()

In [29]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transforms,
    batch_size=32,
)

In [30]:
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7ae22fa15c70>,
 ['pizza', 'steak', 'sushi'])

## 3. Setting up a pre-trained model

In [31]:
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 118MB/s]


In [35]:
## print a summary
from torchinfo import summary

summary(model=model,
        input_size=(32,3,224,224), # example of [bacth_size, colour_channels, height, width]
        col_names = ["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

### 3.2. Freezing the base layers.

In [41]:
for param in model.features.parameters():
  param.requires_grad= False

In [42]:
set_seeds(42)

model.classifier = nn.Sequential(
    nn.Dropout(p=0.2,inplace=True),
    nn.Linear(in_features=1280,
              out_features=len(class_names))
).to(device)

model.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)

In [43]:
summary(model=model,
        input_size=(32,3,224,224),
        col_names= ["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 